# Ablation and steering experiments
We're going to
- Load the data for helix face value bins on train sequences
- Load all of our test proteins, randomly sample 1M residues (around 11% of the total set across 72k proteins, or 98% of proteins in the test set)
- Iterate through our test set by randomly picking residues, then doing three different steering experiments:
    1. Mask residue. On layer 18, subtract the mean activation for a residue's current angle, then add the global mean back
        - Controls: subtract mean for random activation instead of bin's own, subtract random vector of same norm
    2. Fit OLS on mean activations, subtract the OLS fitted activation for current value then add back the fitted value for the global mean
        - Controls: subtract fitted activation of random value instead of residue's own, subtract random vector of same norm
    3. Same as above but fit a + b \cos \theta + c \sin \theta instead of OLS
- Endpoints:
    -   Log likelihood of the correct aka. original aminoacid
    -   Top-1 accuracy (is the top prediction changing ?)
    -   95% confidence intervals from each, computed from deciles for effect 

In [1]:
import json

import numpy as np
import polars as pl
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer, EsmForMaskedLM

from biomanifolds import config
from biomanifolds.datasets import swissprot
from biomanifolds.labels import af2_structure_labels

LAYERS = [28, 30]  # layer 18 had no effect in the 10k-residue run (results in steering/nb03)
MODEL = "facebook/esm2_t30_150M_UR50D"
N_RESIDUES = 1_000_000
AA = "ACDEFGHIKLMNPQRSTVWY"  # order of the saved amino-acid log-probabilitiesso 
OUT_DIR = config.data_path("steering") / "nb03_1M"  # results of this notebook
OUT_DIR.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(0)

# Bin sums and counts of helix face on the train set, for each layer (from notebook 2): bin k is [k - 180, k - 179) degrees
helix_sums, helix_counts = {}, {}
for layer in LAYERS:
    bin_files = {name: config.data_path("label_bins") / f"train_{name}_L{layer}.npy" for name in [
        "helix_face_1deg_bin_sums", "helix_face_1deg_bin_counts"]}
    assert all(path.exists() for path in bin_files.values()), f"no layer-{layer} bin files found, run notebook 2 first"
    helix_sums[layer], helix_counts[layer] = (np.load(path) for path in bin_files.values())

# ESM-2 with its amino-acid head, to read the log-likelihood of the masked residue
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL, attn_implementation="sdpa").eval().to(device)


# All test proteins
sequences = pl.read_parquet(swissprot.swissprot_dir() / "swissprot.parquet", columns=["accession", "sequence"])
splits = pl.read_parquet(swissprot.swissprot_dir() / "splits" / "splits.parquet", columns=["accession", "split"])
test_proteins = (
    sequences.join(splits.filter(pl.col("split") == "test"), on="accession")
    .select("accession", "sequence")
    .sort("accession")
)
test_proteins

config.json:   0%|          | 0.00/779 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  595MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/491 [00:00<?, ?it/s]

accession,sequence
str,str
"""A0A017TC06""","""MIDASSLKALVRRAEGGLVEALGLRPLQLD…"
"""A0A023PXA5""","""MLLSELVATASSLPYTAISIHNNCRVPAAR…"
"""A0A023PXB0""","""MFINGFVNYPVRTPPNDLLQVVLHGFLRCP…"
"""A0A023PXB5""","""MFALIISSKGKTSGFFFNSSFSSSALVGIA…"
"""A0A023PXC2""","""MLPLCLTFLSFFLSLGGSFKAVMTKEEADG…"
…,…
"""X0JZ91""","""MIASIRTTRRLSNYNSVIPPPEPLNTDPDM…"
"""X1WER2""","""MATVEELKLRIRELENELIKSKQKQSDAEH…"
"""X4Y2L4""","""MKEIAVTIDDKNVIASVSESFHGVAFDASL…"


## Fitting OLS on activations, as well as activations transformed by harmonic functions

In [2]:
theta = np.deg2rad(np.arange(360) - 179.5)  # centre of each 1-degree bin, in radians
H, global_mean, B_linear, B_harmonic = {}, {}, {}, {}
for layer in LAYERS:
    H[layer] = helix_sums[layer] / helix_counts[layer][:, None]  # (360, 640) mean activation per bin
    global_mean[layer] = helix_sums[layer].sum(0) / helix_counts[layer].sum()  # mu: mean activation over all train helix residues
    weights = np.sqrt(helix_counts[layer])[:, None]  # least squares on the bin means, weighted by the residues per bin

    # OLS on linear inputs
    X = np.column_stack([np.ones(360), theta])
    B_linear[layer] = np.linalg.lstsq(X * weights, H[layer] * weights, rcond=None)[0]

    # OLS on inputs transformed by harmonic functions
    X = np.column_stack([np.ones(360), np.cos(theta), np.sin(theta)])
    B_harmonic[layer] = np.linalg.lstsq(X * weights, H[layer] * weights, rcond=None)[0]

# Mean activation for a specific value of the latent
def bin_mean_activation(t, layer):
    return H[layer][int(np.floor(np.rad2deg(t) + 180)) % 360]

def linear_activation(t, layer):
    return np.array([1.0, t]) @ B_linear[layer]

def harmonic_activation(t, layer):
    return np.array([1.0, np.cos(t), np.sin(t)]) @ B_harmonic[layer]

# Object containing the fitted activations for each angle according to each method
activation_at_angle = {"bin_means": bin_mean_activation, "linear": linear_activation, "harmonic": harmonic_activation}

# weighted R^2 of the two fits on the bin means, for each layer
r2 = []
for layer in LAYERS:
    total = (helix_counts[layer][:, None] * (H[layer] - global_mean[layer]) ** 2).sum()
    for name, activation in [("linear", linear_activation), ("harmonic", harmonic_activation)]:
        fitted = np.stack([activation(t, layer) for t in theta])
        r2.append({"layer": layer, "fit": name, "r2": 1 - (helix_counts[layer][:, None] * (H[layer] - fitted) ** 2).sum() / total})
pl.DataFrame(r2)

layer,fit,r2
i64,str,f64
28,"""linear""",0.168054
28,"""harmonic""",0.953803
30,"""linear""",0.114587
30,"""harmonic""",0.958772


## Randomly sampling 1M residues from the test dataset

In [3]:
residues = (
    af2_structure_labels.load_labels(["accession", "i", "helix_face"])
    .filter(pl.col("helix_face").is_not_null())
    .join(test_proteins.select("accession").lazy(), on="accession")
    .collect(engine="streaming")
    .sort("accession", "i")  # fixed order, so that the sample is reproducible
    .sample(N_RESIDUES, seed=0)
    .sort("accession", "i")  # grouped by protein, for the edit loop
)
residues

accession,i,helix_face
str,i32,f32
"""A0A024SC78""",40,-19.566528
"""A0A024SC78""",68,-18.572887
"""A0A024SC78""",139,161.152023
"""A0A024SC78""",242,36.43911
"""A0A024SH76""",148,134.54187
…,…,…
"""X4Y2L4""",351,157.112289
"""X5HYT8""",135,-95.648308
"""X5HYT8""",143,-57.476585


## Editing activations
For each randomly sampled residue, ablate the fitted latent vector for its angle according to each different method, measure the effect on logits and the top-predicted aminoacid.

In [4]:
SHARD_PROTEINS = 1000  # proteins per results file; finished files are skipped, so the cell can run again after a crash
MAX_TOKENS = 2**17  # tokens in one forward pass of the edited copies
EDITS = ["none"] + [f"{method}{control}" for method in activation_at_angle for control in ["", " | random angle", " | random vector"]]
AA_IDS = tokenizer.convert_tokens_to_ids(list(AA))  # vocabulary ids of the 20 amino acids
VOCABULARY = np.array(tokenizer.convert_ids_to_tokens(list(range(len(tokenizer)))))
N_LAYERS = len(model.esm.encoder.layer)


def run_layers(h, start, end):
    """Run ESM layers start+1 .. end on the hidden states h (batch, tokens, 640)."""
    rotary = model.esm.rotary_embeddings(h, torch.arange(h.shape[1], device=h.device)[None])  # rotary position embeddings
    for block in model.esm.encoder.layer[start:end]:
        h = block(h, position_embeddings=rotary)
    return h


def edit_vectors(n, face):
    """The changes added to h at the masked position, {layer: (edits, 640)}: mu - f(theta), and the random angle.

    One random generator per residue (n: its row in `residues`), so that every edit can be computed again:
    draws in order: the random angle, then for each layer and method one random direction (640,)."""
    residue_rng = np.random.default_rng([0, n])
    own, random_angle = np.deg2rad(face), residue_rng.uniform(-np.pi, np.pi)
    deltas = {}
    for layer in LAYERS:
        d = {"none": np.zeros(640)}
        for method, activation in activation_at_angle.items():
            d[method] = global_mean[layer] - activation(own, layer)
            d[f"{method} | random angle"] = global_mean[layer] - activation(random_angle, layer)
            random_direction = residue_rng.standard_normal(640)
            d[f"{method} | random vector"] = random_direction / np.linalg.norm(random_direction) * np.linalg.norm(d[method])
        deltas[layer] = np.stack([d[name] for name in EDITS])
    return deltas, np.rad2deg(random_angle)


def edit_protein(sequence, ns, positions, faces):
    """All edits for the sampled residues of one protein; the masked copies of the protein run in one batch."""
    tokens = tokenizer([sequence], return_tensors="pt")["input_ids"].to(device)  # (1, tokens)
    per_chunk = max(1, MAX_TOKENS // (len(EDITS) * tokens.shape[1]))
    blocks = []
    for c in range(0, len(ns), per_chunk):
        n, face = ns[c:c + per_chunk], faces[c:c + per_chunk]
        position = torch.tensor(positions[c:c + per_chunk], device=device) + 1  # token 0 is <cls>
        rows = torch.arange(len(n), device=device)
        original = tokens[0, position]
        input_ids = tokens.repeat(len(n), 1)
        input_ids[rows, position] = tokenizer.mask_token_id
        vectors = [edit_vectors(*residue) for residue in zip(n, face)]

        # the unedited stream runs once, layer by layer; at each layer in LAYERS the edits branch off and run to the last layer
        with torch.no_grad():
            h = model.esm.embeddings(input_ids=input_ids, attention_mask=torch.ones_like(input_ids))
            previous = 0
            for layer in LAYERS:
                h = run_layers(h, previous, layer)  # unedited, up to `layer`
                previous = layer
                edited = h.repeat_interleave(len(EDITS), 0)  # row r * len(EDITS) + e: residue r, edit e
                edited_rows, edited_position = torch.arange(len(edited), device=device), position.repeat_interleave(len(EDITS))
                edited[edited_rows, edited_position] += torch.tensor(np.concatenate([d[layer] for d, _ in vectors]), dtype=torch.float32, device=device)
                edited = run_layers(edited, layer, N_LAYERS)
                log_probs = model.lm_head(model.esm.encoder.emb_layer_norm_after(edited[edited_rows, edited_position])).log_softmax(-1)  # (rows, vocabulary)
                blocks.append(pl.DataFrame({
                    "n": np.repeat(n, len(EDITS)), "layer": layer, "edit": EDITS * len(n),
                    "random_angle": np.repeat([angle for _, angle in vectors], len(EDITS)),
                    "edit_norm": np.linalg.norm(np.concatenate([d[layer] for d, _ in vectors]), axis=1),
                    "log_likelihood": log_probs[edited_rows, original.repeat_interleave(len(EDITS))].cpu().numpy(),
                    "top1_id": log_probs.argmax(-1).cpu().numpy(),
                    "aa_log_probs": log_probs[:, AA_IDS].float().cpu().numpy(),  # log-probabilities of the 20 amino acids, in the order of AA
                }))
    return pl.concat(blocks)


# one row per protein, with its sampled residues; n: the row of the residue in `residues`
proteins = (
    residues.with_row_index("n").group_by("accession", maintain_order=True).agg("n", "i", "helix_face")
    .join(test_proteins, on="accession", maintain_order="left")
)
(OUT_DIR / "results").mkdir(exist_ok=True)
for shard, start in enumerate(tqdm(range(0, len(proteins), SHARD_PROTEINS), desc="shards")):
    path = OUT_DIR / "results" / f"shard-{shard:05d}.parquet"
    if path.exists():
        continue
    rows = pl.concat([edit_protein(sequence, ns, positions, faces)
                      for _, ns, positions, faces, sequence in proteins[start:start + SHARD_PROTEINS].iter_rows()])
    rows = (
        rows.join(residues.with_row_index("n").join(test_proteins, on="accession"), on="n", maintain_order="left")
        .with_columns(
            sequence_length=pl.col("sequence").str.len_chars(), aa=pl.col("sequence").str.slice(pl.col("i"), 1),
            method=pl.col("edit").str.split(" | ").list.first(),
            control=pl.when(pl.col("edit") == "none").then(pl.lit("none"))
                     .otherwise(pl.col("edit").str.split(" | ").list.get(1, null_on_oob=True).fill_null("own angle")),
            top1_aa=pl.Series(VOCABULARY[rows["top1_id"].to_numpy()]),
        )
        .with_columns(
            angle_used=pl.when(pl.col("control") == "own angle").then(pl.col("helix_face"))
                        .when(pl.col("control") == "random angle").then(pl.col("random_angle")),
            top1_correct=pl.col("top1_aa") == pl.col("aa"),
        )
        .select("accession", "i", "sequence_length", "aa", "helix_face", "layer", "edit", "method", "control", "angle_used",
                "edit_norm", "log_likelihood", "top1_correct", "top1_aa", "aa_log_probs")
    )
    rows.write_parquet(path.with_suffix(".tmp"))
    path.with_suffix(".tmp").rename(path)  # a shard file exists only when it is complete

results = pl.read_parquet(OUT_DIR / "results" / "*.parquet")
(OUT_DIR / "metadata.json").write_text(json.dumps({
    "model": MODEL, "layers": LAYERS, "n_residues": N_RESIDUES, "residue_sample_seed": 0,
    "residue_rng": "np.random.default_rng([0, n]), n = row of the residue in `residues`; draws: random angle, then per layer and method a random direction",
    "aa_log_probs_order": AA, "bin_files": "label_bins: train_helix_face_1deg_bin_{sums,counts}_L{layer}.npy",
}, indent=1))
results

shards:   0%|          | 0/73 [00:00<?, ?it/s]

accession,i,sequence_length,aa,helix_face,layer,edit,method,control,angle_used,edit_norm,log_likelihood,top1_correct,top1_aa,aa_log_probs
str,i32,u32,str,f32,i32,str,str,str,f64,f64,f32,bool,str,"array[f32, 20]"
"""A0A024SC78""",40,248,"""L""",-19.566528,28,"""none""","""none""","""none""",null,0.0,-1.75709,false,"""A""","[-1.186685, -4.087124, … -5.122841]"
"""A0A024SC78""",40,248,"""L""",-19.566528,28,"""bin_means""","""bin_means""","""own angle""",-19.566528,18.699353,-1.807129,false,"""A""","[-1.183759, -4.03489, … -5.044221]"
"""A0A024SC78""",40,248,"""L""",-19.566528,28,"""bin_means | random angle""","""bin_means""","""random angle""",49.306207,13.5615,-1.761129,false,"""A""","[-1.20747, -4.256356, … -5.017302]"
"""A0A024SC78""",40,248,"""L""",-19.566528,28,"""bin_means | random vector""","""bin_means""","""random vector""",null,18.699353,-1.743749,false,"""A""","[-1.225015, -4.122052, … -5.155508]"
"""A0A024SC78""",40,248,"""L""",-19.566528,28,"""linear""","""linear""","""own angle""",-19.566528,1.307919,-1.758741,false,"""A""","[-1.184332, -4.067733, … -5.133151]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""X5HYT8""",178,425,"""R""",179.99881,30,"""linear | random angle""","""linear""","""random angle""",39.700172,3.205276,-2.123972,false,"""K""","[-2.568397, -4.537205, … -4.146075]"
"""X5HYT8""",178,425,"""R""",179.99881,30,"""linear | random vector""","""linear""","""random vector""",null,14.874605,-2.117405,false,"""K""","[-2.583311, -4.531467, … -4.161912]"
"""X5HYT8""",178,425,"""R""",179.99881,30,"""harmonic""","""harmonic""","""own angle""",179.99881,31.889584,-2.1833,false,"""K""","[-2.490525, -4.197612, … -3.914175]"


## Effects
Change of each endpoint against the unedited pass of the same residue, with bootstrap 95% confidence intervals of the mean.

In [5]:
from IPython.display import display

N_BOOT = 2000

unedited = results.filter(pl.col("edit") == "none").select("accession", "i", "layer", ll_unedited="log_likelihood", top1_unedited="top1_correct")
effects = results.drop("aa_log_probs").join(unedited, on=["accession", "i", "layer"]).with_columns(
    delta_ll=pl.col("log_likelihood") - pl.col("ll_unedited"),
    delta_top1=pl.col("top1_correct").cast(pl.Float64) - pl.col("top1_unedited").cast(pl.Float64),
)

# Bootstrap 95% confidence intervals of the mean, resampling proteins: in each bootstrap sample, every protein gets a multinomial weight
proteins = effects["accession"].unique().sort()
boot_weights = rng.multinomial(len(proteins), np.full(len(proteins), 1 / len(proteins)), size=N_BOOT).astype(np.float64)

def bootstrap_ci(group, column):
    per_protein = pl.DataFrame({"accession": proteins}).join(
        group.group_by("accession").agg(s=pl.col(column).sum(), n=pl.len()), on="accession", how="left").fill_null(0)
    means = (boot_weights @ per_protein["s"].to_numpy()) / (boot_weights @ per_protein["n"].to_numpy())
    return np.percentile(means, [2.5, 97.5])

summary = []
for (layer, name), group in effects.group_by("layer", "edit", maintain_order=True):
    ll_low, ll_high = bootstrap_ci(group, "delta_ll")
    top1_low, top1_high = bootstrap_ci(group, "delta_top1")
    summary.append({"layer": layer, "edit": name, "mean_ll_correct": group["log_likelihood"].mean(),  # mean log p of the correct residue, after the edit
                    "mean_delta_ll": group["delta_ll"].mean(), "ll_ci_low": ll_low, "ll_ci_high": ll_high,
                    "top1_accuracy": group["top1_correct"].mean(), "mean_delta_top1": group["delta_top1"].mean(),
                    "top1_ci_low": top1_low, "top1_ci_high": top1_high,
                    "arith_mean_p_correct": group["log_likelihood"].exp().mean()})  # arithmetic mean probability of the correct residue
summary = pl.DataFrame(summary)
summary.write_parquet(OUT_DIR / "summary.parquet")
pl.Config.set_tbl_rows(-1)  # show every row and column
pl.Config.set_tbl_cols(-1)
for layer in LAYERS:
    display(summary.filter(pl.col("layer") == layer))

layer,edit,mean_ll_correct,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high,arith_mean_p_correct
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
28,"""none""",-1.770799,0.0,0.0,0.0,0.438242,0.0,0.0,0.0,0.315929
28,"""bin_means""",-1.780834,-0.010034,-0.010271,-0.009797,0.436171,-0.002071,-0.002383,-0.001758,0.3141
28,"""bin_means | random angle""",-1.777388,-0.006589,-0.006823,-0.00637,0.437001,-0.001241,-0.00154,-0.00094,0.314188
28,"""bin_means | random vector""",-1.773064,-0.002264,-0.002385,-0.002135,0.437825,-0.000417,-0.000668,-0.00016,0.315383
28,"""linear""",-1.771586,-0.000787,-0.000857,-0.000718,0.43804,-0.000202,-0.00038,-0.000027,0.315819
28,"""linear | random angle""",-1.771482,-0.000683,-0.000754,-0.000612,0.438195,-0.000047,-0.000211,0.000127,0.315604
28,"""linear | random vector""",-1.771157,-0.000358,-0.000407,-0.000307,0.438243,0.000001,-0.000155,0.000157,0.315839
28,"""harmonic""",-1.780592,-0.009793,-0.010027,-0.009561,0.436222,-0.00202,-0.002334,-0.001717,0.314159
28,"""harmonic | random angle""",-1.777271,-0.006472,-0.006702,-0.006248,0.436921,-0.001321,-0.001628,-0.001015,0.314221


layer,edit,mean_ll_correct,mean_delta_ll,ll_ci_low,ll_ci_high,top1_accuracy,mean_delta_top1,top1_ci_low,top1_ci_high,arith_mean_p_correct
i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64
30,"""none""",-1.770799,0.0,0.0,0.0,0.438242,0.0,0.0,0.0,0.315929
30,"""bin_means""",-1.798345,-0.027546,-0.02795,-0.027157,0.433668,-0.004574,-0.004958,-0.004195,0.310686
30,"""bin_means | random angle""",-1.791409,-0.02061,-0.021005,-0.020212,0.434803,-0.003439,-0.003805,-0.003057,0.313184
30,"""bin_means | random vector""",-1.772261,-0.001462,-0.001566,-0.001361,0.437911,-0.000331,-0.000562,-0.000107,0.315701
30,"""linear""",-1.771256,-0.000457,-0.000512,-0.000403,0.438212,-0.00003,-0.000174,0.00012,0.315767
30,"""linear | random angle""",-1.771214,-0.000414,-0.000469,-0.000359,0.438224,-0.000018,-0.000166,0.000131,0.315863
30,"""linear | random vector""",-1.770945,-0.000146,-0.000182,-0.000112,0.438146,-0.000096,-0.000227,0.000033,0.315906
30,"""harmonic""",-1.798035,-0.027236,-0.027625,-0.02685,0.433691,-0.004551,-0.004929,-0.004167,0.310741
30,"""harmonic | random angle""",-1.791258,-0.020459,-0.020858,-0.020058,0.434812,-0.00343,-0.003794,-0.003061,0.313202
